# DataStax Astra DB Vector Store & Enterprise Cassandra Architecture

## 1. Overview and Enterprise Motivation
**DataStax Astra DB** is a serverless, multi-cloud vector database built directly on **Apache Cassandra**:
- **Peer-to-Peer Distributed Architecture**: Masterless ring topology guaranteeing 99.999% uptime with active-active multi-region replication.
- **Storage-Attached Indexing (SAI)**: Deeply integrated vector indexing that lives directly alongside Cassandra's immutable SSTables on disk.
- **JVector Indexing Engine**: Uses a graph-based vector search engine (inspired by DiskANN and Vamana graphs) engineered specifically for out-of-core, high-concurrency search.
- **Unified Hybrid Search**: Seamlessly executes high-dimensional vector similarity alongside structured relational CQL metadata filtering in a single query pass.

---

## 2. Architecture & Vector Retrieval Flow

![Vector Stores Architecture](workflow_vectorstores.png)

## 3. Technology Stack
- **Embeddings Model**: HuggingFace `sentence-transformers/all-MiniLM-L6-v2` (384-dimensional dense vectors).
- **Vector Store**: `langchain_astradb.AstraDBVectorStore` (Apache Cassandra).
- **Language Model**: Groq `openai/gpt-oss-120b` via `init_chat_model('groq:openai/gpt-oss-120b')`.

In [ ]:
# Step 1: Install official LangChain DataStax AstraDB and Groq integration packages
!pip install -qU langchain langchain-core langchain-astradb langchain-groq langchain-huggingface sentence-transformers

In [ ]:
# Step 2: Environment Configuration & API Keys
import os
import warnings
warnings.filterwarnings('ignore')

from dotenv import load_dotenv, find_dotenv

# Load variables from .env
load_dotenv('../.env')
load_dotenv(find_dotenv())

# Astra DB Credentials from https://astra.datastax.com
ASTRA_DB_API_ENDPOINT = os.getenv("ASTRA_DB_API_ENDPOINT", "https://your-astra-db-id-region.apps.astra.datastax.com")
ASTRA_DB_APPLICATION_TOKEN = os.getenv("ASTRA_DB_APPLICATION_TOKEN", "AstraCS:your-token-here")

# Groq API Key
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

print("Environment Configuration:")
print(f"- Astra Endpoint Configured: {bool(os.getenv('ASTRA_DB_API_ENDPOINT'))}")
print(f"- Astra Token Configured:    {bool(os.getenv('ASTRA_DB_APPLICATION_TOKEN'))}")
print(f"- Groq API Key Configured:   {bool(GROQ_API_KEY)}")

In [ ]:
# Step 3: Initialize HuggingFace Embeddings & Groq LLM
try:
    from langchain_huggingface import HuggingFaceEmbeddings
except ImportError:
    from langchain_community.embeddings import HuggingFaceEmbeddings

from langchain.chat_models import init_chat_model

# 1. Embeddings: HuggingFace sentence-transformers/all-MiniLM-L6-v2 (384 dimensions)
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
print("HuggingFace all-MiniLM-L6-v2 embedding model loaded (384 dimensions).")

# 2. LLM: Groq ultra-fast LPU inference
try:
    llm = init_chat_model('groq:openai/gpt-oss-120b')
    test_res = llm.invoke("Respond with 'AstraDB Ready' in 2 words.")
    print(f"Groq LLM Status: {test_res.content}")
except Exception as e:
    print(f"Fallback to groq:llama-3.3-70b-versatile due to: {e}")
    llm = init_chat_model('groq:llama-3.3-70b-versatile')

In [ ]:
# Step 4: Connect to DataStax Astra DB Vector Store
# When live credentials are provided in .env, connects directly to your cloud Cassandra collection.
# If credentials are placeholders, provides a local transparent fallback so the entire workflow executes cleanly.

COLLECTION_NAME = "astra_vector_langchain"

has_valid_credentials = (
    "your-astra-db" not in ASTRA_DB_API_ENDPOINT and 
    "your-token" not in ASTRA_DB_APPLICATION_TOKEN and 
    bool(os.getenv("ASTRA_DB_APPLICATION_TOKEN"))
)

if has_valid_credentials:
    try:
        from langchain_astradb import AstraDBVectorStore
        vector_store = AstraDBVectorStore(
            embedding=embedding_model,
            api_endpoint=ASTRA_DB_API_ENDPOINT,
            collection_name=COLLECTION_NAME,
            token=ASTRA_DB_APPLICATION_TOKEN,
            namespace=None,
        )
        print(f"Connected live to DataStax Astra DB collection '{COLLECTION_NAME}'!")
    except Exception as e:
        print(f"Live AstraDB connection notice ({e}). Initializing local fallback vectorstore.")
        from langchain_core.vectorstores import InMemoryVectorStore
        vector_store = InMemoryVectorStore(embedding=embedding_model)
else:
    print("No live Astra DB credentials found in .env; utilizing local emulation for interactive exploration.")
    from langchain_core.vectorstores import InMemoryVectorStore
    vector_store = InMemoryVectorStore(embedding=embedding_model)

vector_store

In [ ]:
# Step 5: Define structured enterprise documents with rich metadata
from langchain_core.documents import Document

documents = [
    Document(
        page_content="I had chocolate chip pancakes and scrambled eggs for breakfast this morning.",
        metadata={"source": "tweet", "category": "food", "author": "alice"},
    ),
    Document(
        page_content="The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.",
        metadata={"source": "news", "category": "weather", "author": "national_weather"},
    ),
    Document(
        page_content="Building an exciting new production RAG system with LangChain and DataStax Astra DB!",
        metadata={"source": "tweet", "category": "tech", "author": "dev_advocate"},
    ),
    Document(
        page_content="Robbers broke into the downtown city bank and stole $1 million in cash overnight.",
        metadata={"source": "news", "category": "crime", "author": "metro_police"},
    ),
    Document(
        page_content="Wow! That sci-fi thriller was an amazing movie. The direction and soundtrack were top notch.",
        metadata={"source": "tweet", "category": "entertainment", "author": "cinema_buff"},
    ),
    Document(
        page_content="Is the new flagship smartphone worth the price? Read this full battery and camera review.",
        metadata={"source": "website", "category": "tech", "author": "gadget_review"},
    ),
    Document(
        page_content="The top 10 soccer players in the world right now ranked by goals, assists, and trophies.",
        metadata={"source": "website", "category": "sports", "author": "world_sports"},
    ),
    Document(
        page_content="LangGraph and DataStax Cassandra are the premier frameworks for stateful, multi-agent AI!",
        metadata={"source": "tweet", "category": "tech", "author": "data_engineer"},
    ),
    Document(
        page_content="The stock market fell 500 points today due to fears of inflation and global rate hikes.",
        metadata={"source": "news", "category": "finance", "author": "financial_post"},
    ),
    Document(
        page_content="Apache Cassandra provides linear write scalability and zero single point of failure for AI vector search.",
        metadata={"source": "website", "category": "database", "author": "datastax_blog"},
    ),
]

print(f"Created {len(documents)} enterprise documents ready for insertion.")

In [ ]:
# Step 6: Ingest documents into the AstraDB Vector Store
# This computes HuggingFace embeddings (384 dimensions) and writes them to the collection
inserted_ids = vector_store.add_documents(documents=documents)

print(f"Ingested {len(inserted_ids)} documents into vector store.")

In [ ]:
# Step 7: Basic Similarity Search
query = "What is the weather forecast?"
results = vector_store.similarity_search(query, k=2)

print(f"Query: '{query}'")
print("=" * 60)
for i, res in enumerate(results, 1):
    print(f"Result {i}:")
    print(f"Content:  {res.page_content}")
    print(f"Metadata: {res.metadata}\n")

In [ ]:
# Step 8: Hybrid Query with Metadata Filtering
# In Astra DB, metadata fields are indexed via Storage Attached Indexing (SAI),
# enabling zero-latency filtering combined with high-dimensional vector search.
query = "LangChain provides abstractions to make working with LLMs easy"

# Filter specifically for source = 'tweet'
results = vector_store.similarity_search(
    query,
    k=3,
    filter={"source": "tweet"},
)

print(f"Filtered Search (source=tweet) for query: '{query}'\n")
for res in results:
    print(f'* "{res.page_content}" | Metadata: {res.metadata}')

In [ ]:
# Step 9: Similarity Search with Numerical Scores
query = "Tell me about building AI and LLM applications"

results_with_score = vector_store.similarity_search_with_score(
    query,
    k=3,
    filter={"source": "tweet"},
)

print(f"Similarity with Scores (source=tweet): '{query}'\n")
for res, score in results_with_score:
    print(f'* [Score={score:.4f}] "{res.page_content}" | Metadata: {res.metadata}')

In [ ]:
# Step 10: Advanced LangChain Retrievers with Astra DB

# Option A: Score Threshold Retriever
# Only returns documents whose similarity score exceeds a specified confidence floor
threshold_retriever = vector_store.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"k": 2, "score_threshold": 0.3},
)

# Option B: Maximal Marginal Relevance (MMR) Retriever
# Balances relevance to query with diversity among retrieved documents (avoids redundant chunks)
mmr_retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 2, "fetch_k": 5},
)

print("Retrieving using MMR for query: 'Stealing from the bank is a crime'")
mmr_docs = mmr_retriever.invoke("Stealing from the bank is a crime")
for doc in mmr_docs:
    print(f"- {doc.page_content} (Source: {doc.metadata['source']})")

In [ ]:
# Step 11: End-to-End Enterprise RAG Pipeline using LCEL & Groq LLM
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# Document formatter
def format_docs(docs):
    return "\n\n".join(f"[Source: {doc.metadata.get('source')}]\n{doc.page_content}" for doc in docs)

# Enterprise RAG prompt
astra_rag_prompt = ChatPromptTemplate.from_template("""You are an enterprise AI data assistant powered by DataStax Astra DB and Groq.
Answer the user's question using ONLY the retrieved context below. Be concise, precise, and factually grounded.

Context:
{context}

Question:
{question}

Answer:""")

# Build the LCEL Chain
astra_rag_chain = (
    {
        "context": mmr_retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | astra_rag_prompt
    | llm
    | StrOutputParser()
)

print("Astra DB + Groq LCEL RAG Chain constructed successfully!")

# Test query execution
test_query = "What framework is best for building stateful agentic applications?"
print(f"\nQuery: {test_query}")
print("=" * 60)
answer = astra_rag_chain.invoke(test_query)
print(f"Groq LLM Answer:\n{answer}")

In [ ]:
# Step 12: Production Multi-Query Demonstration Function
def query_astra_rag(question: str):
    print(f"User Query: {question}")
    print("-" * 50)
    
    # 1. Fetch source documents
    sources = mmr_retriever.invoke(question)
    
    # 2. Generate grounded answer
    answer = astra_rag_chain.invoke(question)
    
    print(f"Answer:\n{answer}\n")
    print("Retrieved Context Chunks:")
    for i, doc in enumerate(sources, 1):
        print(f"  [{i}] ({doc.metadata.get('source')}) {doc.page_content[:90]}...")
    print("=" * 70 + "\n")

# Run test questions
benchmark_queries = [
    "What was the financial news about the stock market?",
    "Tell me about the bank robbery investigation.",
    "What are the benefits of Apache Cassandra for AI?"
]

for q in benchmark_queries:
    query_astra_rag(q)